# Module 3 lab: anomalies in NYC taxi demand

**MSCA Doctoral Programme · Module 3 of 3 · Luiza Adelina Ciucu, Dell Technologies.** Parts A–E repeat the lecture on the same series: 10,320 half-hour taxi counts (July 2014 to January 2015) with five labelled events (NYC Marathon, Thanksgiving, Christmas, New Year, a snowstorm). Data: [NAB `nyc_taxi`](https://github.com/numenta/NAB). Every part ends with a question to discuss with your neighbour.

```
pip install dtaianomaly statsmodels scikit-learn pandas matplotlib
```

Library: [ML-KULeuven/dtaianomaly](https://github.com/ML-KULeuven/dtaianomaly) (Carpentier et al., 2025, arXiv:2502.14381).

In [ ]:
import warnings, os, json; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score
from statsmodels.tsa.seasonal import MSTL
from dtaianomaly.evaluation import VolumeUnderPR, PointAdjustedFBeta, RangeBasedFBeta, FBeta
plt.rcParams.update({"figure.figsize": (11, 3), "axes.spines.top": False, "axes.spines.right": False})

URL = "https://raw.githubusercontent.com/numenta/NAB/master/data/realKnownCause/nyc_taxi.csv"
PATH = os.environ.get("NYC_TAXI_CSV", URL)
df = pd.read_csv(PATH, parse_dates=["timestamp"], index_col="timestamp")
x = df["value"].astype(float)
WINDOWS = {"Marathon": ("2014-10-30 15:30", "2014-11-03 22:30"), "Thanksgiving": ("2014-11-25 12:00", "2014-11-29 19:00"), "Christmas": ("2014-12-23 11:30", "2014-12-27 18:30"), "New Year": ("2014-12-29 21:30", "2015-01-03 04:30"), "Snowstorm": ("2015-01-24 20:30", "2015-01-29 03:30")}
y = np.zeros(len(x), dtype=int)
for a, b in WINDOWS.values(): y[(x.index >= a) & (x.index <= b)] = 1
print(len(x), "points,", int(y.sum()), "labelled (", round(100 * y.mean(), 1), "%)")

In [ ]:
def show(series, title, flags=None, ax=None):
    ax = ax or plt.subplots()[1]
    ax.plot(series.index, series.values, lw=.7, color="#0b2545")
    for a, b in WINDOWS.values(): ax.axvspan(pd.Timestamp(a), pd.Timestamp(b), color="#d6334a", alpha=.2)
    if flags is not None: ax.scatter(series.index[flags], series.values[flags], s=8, color="#d6334a", zorder=3)
    ax.set_title(title); return ax
show(x, "NYC taxi demand per 30 min. Red bands: the five labelled windows"); plt.show()
mu, sd = x.mean(), x.std()
flag = (x - mu).abs().values > 3 * sd
print("global 3-sigma rule flags", int(flag.sum()), "points; events hit:", sum(flag[(x.index >= a) & (x.index <= b)].any() for a, b in WINDOWS.values()), "of 5")

**Discuss.** Which event does the global rule see, and why does a rule that ignores the hour of the week fail on the others?

## Part A · Trend, seasonality, residual (lecture Part 2)

`MSTL` removes a trend and two seasonal cycles: a day (48 half-hours) and a week (336).

In [ ]:
res = MSTL(x, periods=(48, 336), stl_kwargs={"robust": True}).fit()
fig, axs = plt.subplots(4, 1, figsize=(11, 7), sharex=True)
for ax, s, n in zip(axs, (x, res.trend, res.seasonal.sum(axis=1), res.resid), ("observed", "trend", "seasonality (day + week)", "residual")):
    ax.plot(s, lw=.6, color="#0b2545"); ax.set_ylabel(n, fontsize=8)
plt.show()

In [ ]:
resid = res.resid
z_res = ...  # TODO: rolling |z-score| of the residual over a 7-day window (336 steps), using min_periods=48
flag = z_res.values > 4
print("residual rule: flagged", int(flag.sum()), "events hit:", sum(flag[(x.index >= a) & (x.index <= b)].any() for a, b in WINDOWS.values()), "of 5")
show(resid, "MSTL residual, flags in red", flag); plt.show()

**Discuss.** The decomposition used the *whole* series, including the days after each event. Could you run it like this in production? (Hint: leakage.)

## Part B · Sliding-window features (lecture Part 3)

Every feature uses the past only: **shift before any rolling statistic**.

In [ ]:
F = pd.DataFrame({"lag1d": x - x.shift(48),
                  "lag1w": x - x.shift(336),
                  "m1d": ...,   # TODO: mean of the previous 24 h, excluding the current point (shift first!)
                  "s1d": ...})  # TODO: std of the previous 24 h, excluding the current point
F = F.dropna(); print(F.shape); F.head()

In [ ]:
yF = y[len(x) - len(F):]
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(F.lag1d[yF == 0], F.lag1w[yF == 0], s=3, alpha=.3, color="#1f5fd6"); ax.scatter(F.lag1d[yF == 1], F.lag1w[yF == 1], s=4, color="#d6334a")
ax.set_xlabel("lag1d"); ax.set_ylabel("lag1w"); plt.show()

**Discuss.** Why is `shift(1)` before `rolling` essential? Which feature would change most if the window were 7 days instead of 24 h?

## Part C · Dynamic z-score vs. Isolation Forest (lecture Part 4)

In [ ]:
def rollz(s, w=336):
    s = pd.Series(s, index=x.index)
    return ((s - s.rolling(w, min_periods=48).mean()) / s.rolling(w, min_periods=48).std()).abs().fillna(0).values
scores = {"z-score, raw": rollz(x), "z-score, residual": rollz(res.resid), "z-score, vs last week": rollz((x - x.shift(336)).fillna(0))}
ifr = IsolationForest(n_estimators=200, random_state=0).fit(x.values[:, None])
scores["IF, raw count"] = -ifr.score_samples(x.values[:, None])
iso = IsolationForest(n_estimators=200, random_state=0).fit(F)
sF = -iso.score_samples(F)                      # higher = more anomalous
scores["IF, window features"] = np.r_[np.zeros(len(x) - len(F)), sF]
print({k: v.shape for k, v in scores.items()})

**Discuss.** In the lecture, the forest on the raw count was no better than a z-score. Why does adding window features change that, when the algorithm stays the same?

## Part D · Evaluate (lecture Part 5)

Report **AUC-PR**, **VUS-PR** (threshold-free), and **event recall** at a fixed alarm budget (top 2%).

In [ ]:
def evaluate(s, budget=0.02):
    s = np.nan_to_num(np.asarray(s, float)); thr = np.quantile(s, 1 - budget); fl = s > thr
    ev = sum(fl[(x.index >= a) & (x.index <= b)].any() for a, b in WINDOWS.values())
    return dict(AUC_PR=average_precision_score(y, s), VUS_PR=VolumeUnderPR().compute(y, s), events_hit=f"{ev} / 5",
                point_recall=round(float((fl & (y == 1)).sum() / y.sum()), 2))
table = pd.DataFrame({k: evaluate(v) for k, v in scores.items()}).T
table.round(2)

In [ ]:
# Point adjustment: if any point of a window is flagged, the whole window counts. Compare with plain and range-based F1.
s = scores["IF, window features"]; yhat = (s > np.quantile(s, .98)).astype(int)
rnd = (np.random.default_rng(0).random(len(x)) < .02).astype(int)   # a RANDOM detector with the same budget
for name, pred in (("Isolation Forest, window features", yhat), ("random detector, 2% flags", rnd)):
    print(f"{name:34s} plain F1 {FBeta(beta=1).compute(y, pred):.2f} | point-adjusted F1 {PointAdjustedFBeta(beta=1).compute(y, pred):.2f} | range-based F1 {RangeBasedFBeta(beta=1).compute(y, pred):.2f}")

**Discuss.** What does a random detector score under point adjustment, and why? Which of the three numbers would you put in a paper?

## Part E · dtaianomaly detectors and the window size

In [ ]:
from dtaianomaly.anomaly_detection import MatrixProfileDetector, IsolationForest as DIF, KNearestNeighbors, PrincipalComponentAnalysis, RandomDetector
X1 = x.values.reshape(-1, 1)
out = {}
for w in (48, 336):                              # one day vs. one week
    for name, det in {"MatrixProfile": MatrixProfileDetector(window_size=w), "IsolationForest": DIF(window_size=w, random_state=0),
                      "kNN": KNearestNeighbors(window_size=w, n_neighbors=10), "PCA": PrincipalComponentAnalysis(window_size=w),
                      "Random": RandomDetector(seed=0)}.items():
        sc = det.fit(X1).decision_function(X1); out[(name, w)] = VolumeUnderPR().compute(y, sc)
pd.Series(out).unstack().round(3)

**Discuss.** Which window size wins for which detector, and what does a day versus a week mean for this series? No detector wins everywhere (Boniol and Palpanas, 2024): what would you do without labels?

## Wrap-up

1. Normal is a **model**: trend, season, residual say what the series would have done.
2. **Context belongs in the features**: lags and trailing statistics, past only.
3. **The input matters more than the algorithm**: the same forest on raw counts is no better than a z-score.
4. **Judge by events**: five windows make a story, not a benchmark; prefer VUS-PR and report event recall.

⭐ Further: try other NAB series, a different alarm budget, or a causal (online) version of the decomposition.